# Gold: hospital_daily_summary

Builds `dbw_caresync_dev.gold.hospital_daily_summary` (one row per hospital per calendar day with lab activity) from `silver.lab_results`, per STTM `sttm_silver_to_gold.xlsx` (sheet `hospital_daily_summary`).

* **Driving table:** `silver.lab_results` grouped by `hospital_id`, `test_date` (no filters)
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`; `silver.calendar` on `date_key = date_format(test_date, 'yyyyMMdd')`
* **Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date` (previous day *with tests* for that hospital)
* **Load:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`
* **Schema note:** `silver.calendar.date_key` is `int`, so the formatted date string is cast to `INT` for the join.

In [0]:
%sql
-- FULL rebuild: one row per hospital per day (driving table silver.lab_results, LEFT JOINs only)
CREATE OR REPLACE TABLE dbw_caresync_dev.gold.hospital_daily_summary AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                AS summary_date,
    COUNT(*)                                 AS total_tests,
    COUNT(DISTINCT patient_id)               AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM dbw_caresync_dev.silver.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  t.hospital_id,
  h.hospital_name,
  t.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)                                                   AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                        AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                           AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                           AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                           AS critical_test_count,
  CAST(t.abnormal_test_count * 100.0 / t.total_tests AS DECIMAL(5,2))          AS abnormal_rate_pct,
  CAST(t.critical_test_count * 100.0 / t.total_tests AS DECIMAL(5,2))          AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                          AS prev_day_total_tests,
  CASE
    WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
    ELSE CAST((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests AS DECIMAL(6,2))
  END                                                                          AS test_volume_change_pct
FROM trend t
LEFT JOIN dbw_caresync_dev.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN dbw_caresync_dev.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)

In [0]:
# Runs only after the CREATE OR REPLACE TABLE cell above succeeds; returns the Gold row count to ADF (runOutput)
target_table = "dbw_caresync_dev.gold.hospital_daily_summary"
record_count = spark.table(target_table).count()
print(f"SUCCESS: {record_count} record(s) in {target_table}")
dbutils.notebook.exit(str(record_count))